# PyTorch → TFLite export and inspection

This notebook exports:

- a **float32** TFLite model
- a **quantized/int8 candidate** TFLite model

Both outputs are saved under `model_files/` in separate folders.

It also inspects both exported models with the installed **LiteRT interpreter** (`ai_edge_litert`) and reports:

- input/output shapes
- input/output dtypes
- quantization parameters
- whether the interface is actually `int8`

> Important: the quantized export path depends on the conversion backend and model support. This notebook **verifies** the result instead of assuming quantization succeeded.


In [17]:

from pathlib import Path
import json
import shutil
import numpy as np
import torch
import torch.nn as nn

from model import DSCNN

import litert_torch
from ai_edge_litert.interpreter import Interpreter

print("torch:", torch.__version__)
print("litert_torch imported successfully")
print("ai_edge_litert imported successfully")


torch: 2.9.1+cu128
litert_torch imported successfully
ai_edge_litert imported successfully


In [18]:

# =========================
# Configuration
# =========================

# Adjust these paths to your project
CHECKPOINT_PATH = "model_files/kws_model.pt"   # e.g. "./checkpoints/best_model.pth"
MODEL_ROOT = Path("model_files")

# Input shape for export (NCHW as used by PyTorch)
# Old setup:
N_MELS = 64
TIME_FRAMES = 101

# Smaller embedded-friendly setup would be:
# N_MELS = 40
# TIME_FRAMES = 49

NUM_CLASSES = 4

# Output layout
FLOAT_DIR = MODEL_ROOT / "float32"
INT8_DIR = MODEL_ROOT / "int8"

FLOAT_TFLITE_PATH = FLOAT_DIR / "model_float32.tflite"
INT8_TFLITE_PATH = INT8_DIR / "model_int8_candidate.tflite"

# Extra artifacts
SAVE_C_ARRAYS = False

# Set True if you want the notebook to fail when the quantized export
# is not actually int8 at the TFLite interface.
REQUIRE_INT8_IO = False

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
FLOAT_DIR.mkdir(parents=True, exist_ok=True)
INT8_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint:", CHECKPOINT_PATH)
print("Model root :", MODEL_ROOT.resolve())
print("Float out  :", FLOAT_TFLITE_PATH.resolve())
print("Int8 out   :", INT8_TFLITE_PATH.resolve())
print("Input shape:", (1, 1, N_MELS, TIME_FRAMES))


Checkpoint: model_files/kws_model.pt
Model root : /home/tui/DHBW/Studienarbeit/coffeemachine-automatization/reduced_Code/model_files
Float out  : /home/tui/DHBW/Studienarbeit/coffeemachine-automatization/reduced_Code/model_files/float32/model_float32.tflite
Int8 out   : /home/tui/DHBW/Studienarbeit/coffeemachine-automatization/reduced_Code/model_files/int8/model_int8_candidate.tflite
Input shape: (1, 1, 64, 101)


## Model definition
Replace this cell with your exact training model if it differs.

In [19]:
def load_checkpoint(model: nn.Module, checkpoint_path: str):
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        cleaned[k[7:] if k.startswith("module.") else k] = v

    model.load_state_dict(cleaned, strict=True)
    return model


In [20]:

# Load model and create example input
model = DSCNN(num_classes=NUM_CLASSES)
model = load_checkpoint(model, CHECKPOINT_PATH)
model.eval()

sample_input = torch.randn(1, 1, N_MELS, TIME_FRAMES, dtype=torch.float32)
sample_inputs = (sample_input,)

with torch.no_grad():
    torch_out = model(*sample_inputs).detach().cpu().numpy()

print("Model loaded.")
print("Sample input shape:", tuple(sample_input.shape))
print("PyTorch output shape:", torch_out.shape)


Model loaded.
Sample input shape: (1, 1, 64, 101)
PyTorch output shape: (1, 4)


## Export float32 model

In [21]:

edge_model_float = litert_torch.convert(model, sample_inputs)
edge_model_float.export(str(FLOAT_TFLITE_PATH))

print("Saved float32/candidate export to:", FLOAT_TFLITE_PATH.resolve())


INFO:tensorflow:Assets written to: /tmp/tmpemn9k_cw/assets


INFO:tensorflow:Assets written to: /tmp/tmpemn9k_cw/assets


Saved float32/candidate export to: /home/tui/DHBW/Studienarbeit/coffeemachine-automatization/reduced_Code/model_files/float32/model_float32.tflite


W0000 00:00:1773855346.064271  750975 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773855346.064300  750975 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773855346.064563  750975 reader.cc:83] Reading SavedModel from: /tmp/tmpemn9k_cw
I0000 00:00:1773855346.065173  750975 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773855346.065178  750975 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpemn9k_cw
I0000 00:00:1773855346.071396  750975 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1773855346.073099  750975 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773855346.118701  750975 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpemn9k_cw
I0000 00:00:1773855346.130252  750975 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 65699 microseconds.
I0000 00:00:1773855346.143643  750975

## Export int8 candidate

This cell tries the quantized export path if the installed `litert_torch` build exposes one.

Because the exact quantization API can vary across versions, this cell:

1. checks for a few likely quantization entry points
2. tries them in a controlled way
3. still verifies the final TFLite interface afterward

If none of the known entry points exists, it falls back to exporting a second copy and clearly marks it as **unverified for int8**.


In [22]:

def try_quantized_export(model, sample_inputs, out_path: Path):
    attempts = []

    # 1) Hypothetical direct flag on convert(...)
    attempts.append(("convert(..., quantize='int8')", lambda: litert_torch.convert(model, sample_inputs, quantize="int8")))

    # 2) Hypothetical boolean flag on convert(...)
    attempts.append(("convert(..., quantize=True)", lambda: litert_torch.convert(model, sample_inputs, quantize=True)))

    # 3) Convert first, then quantize method on returned object
    def convert_then_quantize_method():
        em = litert_torch.convert(model, sample_inputs)
        if not hasattr(em, "quantize"):
            raise AttributeError("Returned edge model has no quantize() method")
        return em.quantize("int8")
    attempts.append(("convert(...).quantize('int8')", convert_then_quantize_method))

    # 4) quantize module helper if available
    def module_level_quantize():
        if not hasattr(litert_torch, "quantize"):
            raise AttributeError("litert_torch has no quantize attribute")
        em = litert_torch.convert(model, sample_inputs)
        return litert_torch.quantize(em, mode="int8")
    attempts.append(("litert_torch.quantize(..., mode='int8')", module_level_quantize))

    last_error = None
    for name, fn in attempts:
        try:
            print(f"Trying quantized export path: {name}")
            edge_model = fn()
            edge_model.export(str(out_path))
            print(f"Quantized export path succeeded with: {name}")
            return name, None
        except Exception as e:
            last_error = e
            print(f"  failed: {type(e).__name__}: {e}")

    print("No known quantized export path succeeded. Exporting fallback copy instead.")
    fallback = litert_torch.convert(model, sample_inputs)
    fallback.export(str(out_path))
    return None, last_error


quant_export_method, quant_export_error = try_quantized_export(model, sample_inputs, INT8_TFLITE_PATH)

print("Saved int8 candidate to:", INT8_TFLITE_PATH.resolve())
print("Successful quantized export method:", quant_export_method)
if quant_export_method is None and quant_export_error is not None:
    print("Last quantization error:", repr(quant_export_error))


Trying quantized export path: convert(..., quantize='int8')
  failed: TypeError: convert() got an unexpected keyword argument 'quantize'
Trying quantized export path: convert(..., quantize=True)
  failed: TypeError: convert() got an unexpected keyword argument 'quantize'
Trying quantized export path: convert(...).quantize('int8')
INFO:tensorflow:Assets written to: /tmp/tmpgg69u04v/assets


INFO:tensorflow:Assets written to: /tmp/tmpgg69u04v/assets
W0000 00:00:1773855348.206738  750975 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773855348.206769  750975 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773855348.206894  750975 reader.cc:83] Reading SavedModel from: /tmp/tmpgg69u04v
I0000 00:00:1773855348.207693  750975 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773855348.207700  750975 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpgg69u04v
I0000 00:00:1773855348.214252  750975 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773855348.255994  750975 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpgg69u04v
I0000 00:00:1773855348.267671  750975 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 60789 microseconds.
I0000 00:00:1773855348.376381  750975 flatbuffer_export.cc:4160] Estimated count of arithmetic

  failed: AttributeError: Returned edge model has no quantize() method
Trying quantized export path: litert_torch.quantize(..., mode='int8')
INFO:tensorflow:Assets written to: /tmp/tmpw5eu4z0a/assets


INFO:tensorflow:Assets written to: /tmp/tmpw5eu4z0a/assets
W0000 00:00:1773855350.303798  750975 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773855350.303828  750975 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773855350.303944  750975 reader.cc:83] Reading SavedModel from: /tmp/tmpw5eu4z0a
I0000 00:00:1773855350.304765  750975 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773855350.304771  750975 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpw5eu4z0a
I0000 00:00:1773855350.310743  750975 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773855350.349912  750975 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpw5eu4z0a
I0000 00:00:1773855350.359342  750975 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 55409 microseconds.
I0000 00:00:1773855350.459755  750975 flatbuffer_export.cc:4160] Estimated count of arithmetic

  failed: TypeError: 'module' object is not callable
No known quantized export path succeeded. Exporting fallback copy instead.
INFO:tensorflow:Assets written to: /tmp/tmpg06oe8gy/assets


INFO:tensorflow:Assets written to: /tmp/tmpg06oe8gy/assets


Saved int8 candidate to: /home/tui/DHBW/Studienarbeit/coffeemachine-automatization/reduced_Code/model_files/int8/model_int8_candidate.tflite
Successful quantized export method: None
Last quantization error: TypeError("'module' object is not callable")


W0000 00:00:1773855352.151154  750975 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773855352.151181  750975 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773855352.151286  750975 reader.cc:83] Reading SavedModel from: /tmp/tmpg06oe8gy
I0000 00:00:1773855352.151965  750975 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773855352.151972  750975 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpg06oe8gy
I0000 00:00:1773855352.157518  750975 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773855352.194876  750975 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpg06oe8gy
I0000 00:00:1773855352.204001  750975 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 52739 microseconds.
I0000 00:00:1773855352.301174  750975 flatbuffer_export.cc:4160] Estimated count of arithmetic ops: 73.431 M  ops, equivalently 36.716 M  MACs


## Inspect exported TFLite models

In [23]:

def _normalize_quant_params(qp):
    # ai_edge_litert may return tuples/lists/arrays depending on version
    try:
        scales = qp[0]
        zero_points = qp[1]
        quantized_dimension = qp[2] if len(qp) > 2 else None
    except Exception:
        return {"raw": qp}

    def to_python(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, (list, tuple)):
            return [to_python(v) for v in x]
        if hasattr(x, "item"):
            try:
                return x.item()
            except Exception:
                pass
        return x

    return {
        "scales": to_python(scales),
        "zero_points": to_python(zero_points),
        "quantized_dimension": to_python(quantized_dimension),
    }


def inspect_tflite_model(model_path):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    info = {
        "path": str(model_path),
        "inputs": [],
        "outputs": [],
    }

    for d in input_details:
        info["inputs"].append({
            "name": d.get("name"),
            "shape": d.get("shape").tolist() if hasattr(d.get("shape"), "tolist") else d.get("shape"),
            "dtype": str(d.get("dtype")),
            "quantization": d.get("quantization"),
            "quantization_parameters": _normalize_quant_params(d.get("quantization_parameters")),
        })

    for d in output_details:
        info["outputs"].append({
            "name": d.get("name"),
            "shape": d.get("shape").tolist() if hasattr(d.get("shape"), "tolist") else d.get("shape"),
            "dtype": str(d.get("dtype")),
            "quantization": d.get("quantization"),
            "quantization_parameters": _normalize_quant_params(d.get("quantization_parameters")),
        })

    return interpreter, info


def print_model_info(title, info):
    print("=" * 80)
    print(title)
    print("Path:", info["path"])
    print("--- Inputs ---")
    for i, d in enumerate(info["inputs"]):
        print(f"[{i}] name={d['name']}")
        print("    shape:", d["shape"])
        print("    dtype:", d["dtype"])
        print("    quantization:", d["quantization"])
        print("    quantization_parameters:", d["quantization_parameters"])
    print("--- Outputs ---")
    for i, d in enumerate(info["outputs"]):
        print(f"[{i}] name={d['name']}")
        print("    shape:", d["shape"])
        print("    dtype:", d["dtype"])
        print("    quantization:", d["quantization"])
        print("    quantization_parameters:", d["quantization_parameters"])
    print("=" * 80)


float_interpreter, float_info = inspect_tflite_model(FLOAT_TFLITE_PATH)
int8_interpreter, int8_info = inspect_tflite_model(INT8_TFLITE_PATH)

print_model_info("FLOAT MODEL", float_info)
print_model_info("INT8 CANDIDATE MODEL", int8_info)


FLOAT MODEL
Path: model_files/float32/model_float32.tflite
--- Inputs ---
[0] name=serving_default_args_0:0
    shape: [1, 1, 64, 101]
    dtype: <class 'numpy.float32'>
    quantization: (0.0, 0)
    quantization_parameters: {'raw': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}}
--- Outputs ---
[0] name=StatefulPartitionedCall:0
    shape: [1, 4]
    dtype: <class 'numpy.float32'>
    quantization: (0.0, 0)
    quantization_parameters: {'raw': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}}
INT8 CANDIDATE MODEL
Path: model_files/int8/model_int8_candidate.tflite
--- Inputs ---
[0] name=serving_default_args_0:0
    shape: [1, 1, 64, 101]
    dtype: <class 'numpy.float32'>
    quantization: (0.0, 0)
    quantization_parameters: {'raw': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_si

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [24]:

def is_int8_interface(info):
    in_ok = all("int8" in d["dtype"] for d in info["inputs"])
    out_ok = all("int8" in d["dtype"] for d in info["outputs"])
    return in_ok and out_ok

float_is_int8 = is_int8_interface(float_info)
int8_is_int8 = is_int8_interface(int8_info)

print("float model interface is int8:", float_is_int8)
print("int8 candidate interface is int8:", int8_is_int8)

if REQUIRE_INT8_IO and not int8_is_int8:
    raise RuntimeError(
        "The exported int8 candidate does not expose true int8 input/output tensors. "
        "Quantization was not fully verified at the interface."
    )


float model interface is int8: False
int8 candidate interface is int8: False


## Run a dummy inference on both exports

In [25]:

def make_dummy_input(detail):
    shape = detail["shape"]
    dtype = detail["dtype"]

    if dtype == np.float32:
        return np.zeros(shape, dtype=np.float32)

    if dtype == np.int8:
        zp = 0
        qp = detail.get("quantization_parameters")
        if qp is not None:
            try:
                zero_points = qp["zero_points"] if isinstance(qp, dict) else qp[1]
                if isinstance(zero_points, list):
                    zp = int(zero_points[0]) if zero_points else 0
                elif isinstance(zero_points, np.ndarray):
                    zp = int(zero_points.flat[0]) if zero_points.size > 0 else 0
                else:
                    zp = int(zero_points)
            except Exception:
                pass
        return np.full(shape, zp, dtype=np.int8)

    raise TypeError(f"Unsupported input dtype for dummy inference: {dtype}")


def run_dummy_inference(interpreter, info):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    dummy = make_dummy_input(input_details[0])
    interpreter.set_tensor(input_details[0]["index"], dummy)
    interpreter.invoke()
    y = interpreter.get_tensor(output_details[0]["index"])
    return y

y_float = run_dummy_inference(float_interpreter, float_interpreter.get_input_details())
y_int8 = run_dummy_inference(int8_interpreter, int8_interpreter.get_input_details())

print("Float export dummy output:")
print(y_float)
print("Int8 candidate dummy output:")
print(y_int8)


Float export dummy output:
[[-1.3557773 -1.6245456  4.3067017 -2.7636194]]
Int8 candidate dummy output:
[[-1.3557773 -1.6245456  4.3067017 -2.7636194]]


## Compare float export output against PyTorch

In [26]:

# Compare only for the float export, using the same sample input as during conversion.
float_input_details = float_interpreter.get_input_details()
float_output_details = float_interpreter.get_output_details()

float_in_dtype = float_input_details[0]["dtype"]
x_np = sample_input.detach().cpu().numpy().astype(float_in_dtype)

float_interpreter.set_tensor(float_input_details[0]["index"], x_np)
float_interpreter.invoke()
tflite_float_out = float_interpreter.get_tensor(float_output_details[0]["index"])

max_abs_diff = np.max(np.abs(torch_out - tflite_float_out))
print("PyTorch output:")
print(torch_out)
print("Float TFLite output:")
print(tflite_float_out)
print("max_abs_diff:", float(max_abs_diff))


PyTorch output:
[[ -3.0704246   9.057724  -12.697816   -3.9126637]]
Float TFLite output:
[[ -3.0704257   9.05772   -12.697807   -3.9126613]]
max_abs_diff: 8.58306884765625e-06


## Optional: save model as C arrays for ESP-IDF

In [27]:

def write_c_array(tflite_path: Path, out_dir: Path, array_name: str):
    data = tflite_path.read_bytes()
    cc_path = out_dir / f"{array_name}.cc"
    h_path = out_dir / f"{array_name}.h"

    with open(cc_path, "w", encoding="utf-8") as f:
        f.write('#include <cstdint>\n\n')
        f.write(f'alignas(16) const unsigned char {array_name}[] = {{\n')
        for i, b in enumerate(data):
            if i % 12 == 0:
                f.write("    ")
            f.write(f"0x{b:02x}, ")
            if i % 12 == 11:
                f.write("\n")
        if len(data) % 12 != 0:
            f.write("\n")
        f.write("};\n\n")
        f.write(f"const int {array_name}_len = {len(data)};\n")

    with open(h_path, "w", encoding="utf-8") as f:
        guard = f"{array_name.upper()}_H_"
        f.write(f"#ifndef {guard}\n")
        f.write(f"#define {guard}\n\n")
        f.write("#include <cstdint>\n\n")
        f.write(f"extern const unsigned char {array_name}[];\n")
        f.write(f"extern const int {array_name}_len;\n\n")
        f.write(f"#endif  // {guard}\n")

    return cc_path, h_path


if SAVE_C_ARRAYS:
    float_cc, float_h = write_c_array(FLOAT_TFLITE_PATH, FLOAT_DIR, "g_model_float32")
    int8_cc, int8_h = write_c_array(INT8_TFLITE_PATH, INT8_DIR, "g_model_int8")
    print("Saved C arrays:")
    print(" ", float_cc)
    print(" ", float_h)
    print(" ", int8_cc)
    print(" ", int8_h)
else:
    print("SAVE_C_ARRAYS is False; skipping C array generation.")


SAVE_C_ARRAYS is False; skipping C array generation.


## Save export summary

In [28]:

summary = {
    "checkpoint_path": CHECKPOINT_PATH,
    "float_tflite_path": str(FLOAT_TFLITE_PATH),
    "int8_tflite_path": str(INT8_TFLITE_PATH),
    "input_shape_nchw": [1, 1, N_MELS, TIME_FRAMES],
    "num_classes": NUM_CLASSES,
    "quantized_export_method": quant_export_method,
    "float_interface_is_int8": bool(float_is_int8),
    "int8_candidate_interface_is_int8": bool(int8_is_int8),
    "float_model_info": float_info,
    "int8_model_info": int8_info,
}

summary_path = MODEL_ROOT / "export_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:", summary_path.resolve())
summary


TypeError: Object of type ndarray is not JSON serializable